In [23]:
import pandas as pd
import sys
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# 1. Load the relevant tables
print("Loading datasets...")
chartevents = pd.read_csv('chartevents.csv', low_memory=False)
icustays = pd.read_csv('icustays.csv', low_memory=False)
d_items = pd.read_csv('d_items.csv', low_memory=False)

# sys.exit()
# 2. Generate exactly 300,000 (3 lakhs) data points by sampling from the chart events
print("Sampling 300,000 rows...")
sampled_events = chartevents.sample(n=300000, random_state=42)

# 3. Merge with dictionary items to get the clinical variable labels instead of just numerical IDs
print("Merging with item definitions...")
merged_data = sampled_events.merge(d_items[['itemid', 'label', 'category', 'param_type']], on='itemid', how='left')

# 4. Merge with ICU stays to get context about the admission (like length of stay and care unit)
print("Merging with ICU stay data...")
merged_data = merged_data.merge(icustays[['stay_id', 'first_careunit', 'last_careunit', 'los']], on='stay_id', how='left')

# 5. Save the consolidated dataset
output_filename = 'generated_3lakhs_dataset.csv'
merged_data.to_csv(output_filename, index=False)
print(f"Success! Saved dataset to {output_filename} with shape: {merged_data.shape}")

In [2]:
# df = pd.read_csv('generated_3lakhs_dataset.csv')
merged_data.head()

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning,label,category,param_type,first_careunit,last_careunit,los
0,10020944,29974575,30757476,62802.0,2131-03-06 08:00:00,2131-03-06 07:47:00,225978,On,NaN,NaN,0.0,Slope,Respiratory,Text,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),9.076829
1,10020187,24104168,37509585,65875.0,2169-01-16 07:30:00,2169-01-16 07:36:00,223967,R Femoral,NaN,NaN,0.0,Angio Site # 1,Cardiovascular,Text,Neuro Surgical Intensive Care Unit (Neuro SICU),Neuro Stepdown,5.452662
2,10040025,27996267,36107367,63894.0,2148-01-26 22:30:00,2148-01-26 23:04:00,228868,2 Person Assist,NaN,NaN,0.0,Assistance,Treatments,Text,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),6.538102
3,10039708,28258130,33281088,25898.0,2140-02-03 08:04:00,2140-02-03 10:04:00,229326,Yes,1.0,NaN,0.0,CAM-ICU MS Change,Pain/Sedation,Text,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),16.180787
4,10037861,24540843,34531557,85013.0,2117-03-24 12:00:00,2117-03-24 13:43:00,227969,Pain evaluated and treated,NaN,NaN,0.0,Safety Measures,Restraint/Support Systems,Text,Neuro Surgical Intensive Care Unit (Neuro SICU),Neuro Surgical Intensive Care Unit (Neuro SICU),10.416782


In [3]:
merged_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 17 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   subject_id      300000 non-null  int64  
 1   hadm_id         300000 non-null  int64  
 2   stay_id         300000 non-null  int64  
 3   caregiver_id    289128 non-null  float64
 4   charttime       300000 non-null  object 
 5   storetime       299494 non-null  object 
 6   itemid          300000 non-null  int64  
 7   value           290669 non-null  object 
 8   valuenum        115642 non-null  float64
 9   valueuom        72986 non-null   object 
 10  warning         299494 non-null  float64
 11  label           300000 non-null  object 
 12  category        300000 non-null  object 
 13  param_type      300000 non-null  object 
 14  first_careunit  300000 non-null  object 
 15  last_careunit   300000 non-null  object 
 16  los             300000 non-null  float64
dtypes: float64

In [3]:
chartevents.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 668862 entries, 0 to 668861
Data columns (total 11 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   subject_id    668862 non-null  int64  
 1   hadm_id       668862 non-null  int64  
 2   stay_id       668862 non-null  int64  
 3   caregiver_id  644622 non-null  float64
 4   charttime     668862 non-null  object 
 5   storetime     667703 non-null  object 
 6   itemid        668862 non-null  int64  
 7   value         648132 non-null  object 
 8   valuenum      257474 non-null  float64
 9   valueuom      162571 non-null  object 
 10  warning       667703 non-null  float64
dtypes: float64(3), int64(4), object(4)
memory usage: 56.1+ MB


In [ ]:
chartevents['value'].value_counts()

dtype('O')

In [9]:
chartevents['value'].nunique()

4241

In [10]:
chartevents.count()

subject_id      668862
hadm_id         668862
stay_id         668862
caregiver_id    644622
charttime       668862
storetime       667703
itemid          668862
value           648132
valuenum        257474
valueuom        162571
warning         667703
dtype: int64

In [13]:
chartevents['warning'].value_counts(normalize=True)*100

warning
0.0    97.814597
1.0     2.185403
Name: proportion, dtype: float64

In [16]:
chartevents['valueuom'].isnull().sum()
d_items['label'].value_counts()

label
Safety Measures                        3
GI #2 Tube Place Method                3
GI #1 Tube Place Method                3
GI #3 Tube Place Method                3
CAM-ICU Inattention                    3
                                      ..
Flow Sensor repositioned (CH)          1
Flow Sensor repositioned (ECMO)        1
Circuit Configuration (ECMO)           1
Emergency Equipment at bedside (CH)    1
Pump plugged into RED outlet (ECMO)    1
Name: count, Length: 3888, dtype: int64

In [25]:
print("Loading datasets...")
chartevents = pd.read_csv('chartevents.csv', low_memory=False)
inputevents = pd.read_csv('inputevents.csv', low_memory=False)
outputevents = pd.read_csv('outputevents.csv', low_memory=False)
icustays = pd.read_csv('icustays.csv', low_memory=False)
d_items = pd.read_csv('d_items.csv', low_memory=False)

print("Extracting and standardizing tables...")

df_chart = chartevents[['subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'valuenum']].copy()
df_chart.rename(columns={'charttime': 'timestamp', 'valuenum': 'event_value'}, inplace=True)
df_chart['event_type'] = 'Vital'

# B. Input Events (IV Fluids, Vasopressors, Meds)
# Use 'starttime' as the event time, and 'amount' or 'rate' as the value
df_input = inputevents[['subject_id', 'hadm_id', 'stay_id', 'starttime', 'itemid', 'amount']].copy()
df_input.rename(columns={'starttime': 'timestamp', 'amount': 'event_value'}, inplace=True)
df_input['event_type'] = 'Input'

# C. Output Events (Urine Output, Drains)
df_output = outputevents[['subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'value']].copy()
df_output.rename(columns={'charttime': 'timestamp', 'value': 'event_value'}, inplace=True)
df_output['event_value'] = pd.to_numeric(df_output['event_value'], errors='coerce') # Force text to NaN
df_output['event_type'] = 'Output'

# ==========================================
# 3. COMBINE ALL EVENTS & MAP LABELS
# ==========================================
print("Combining all temporal events...")

# Stack them vertically into one massive long-format dataframe
all_events = pd.concat([df_chart, df_input, df_output], ignore_index=True)

# Drop any rows with missing values or missing timestamps to clean the data
all_events.dropna(subset=['event_value', 'timestamp'], inplace=True)

# Merge with d_items to get human-readable names (Explainable UI requirement)
all_events = all_events.merge(d_items[['itemid', 'label']], on='itemid', how='left')

# ==========================================
# 4. TIME-SERIES SYNCHRONIZATION (Hourly Bins)
# ==========================================
print("Synchronizing timelines (Hourly Resampling)...")

# Convert timestamp to datetime object
all_events['timestamp'] = pd.to_datetime(all_events['timestamp'])

# ROUND DOWN to the nearest hour (e.g., 10:45 AM becomes 10:00 AM)
# This allows us to sync a blood pressure taken at 10:15 with an IV drip started at 10:45
all_events['time_hour'] = all_events['timestamp'].dt.floor('H')

# ==========================================
# 5. PIVOT TO THE FINAL ML MATRIX
# ==========================================
print("Pivoting data to feature matrix...")

# Group by patient stay and the specific hour, then pivot the labels into columns
# We use 'mean' for vitals and 'sum' for inputs/outputs is generally better, 
# but 'mean' works as a universal aggregator for the baseline.
time_series_matrix = all_events.pivot_table(
    index=['subject_id', 'hadm_id', 'stay_id', 'time_hour'],
    columns='label',
    values='event_value',
    aggfunc='mean'
).reset_index()

# ==========================================
# 6. MERGE WITH ICU STAY CONTEXT
# ==========================================
print("Adding ICU Stay boundaries...")

# Only keep the necessary context from icustays
icu_context = icustays[['stay_id', 'intime', 'outtime', 'los']]

final_dataset = time_series_matrix.merge(icu_context, on='stay_id', how='inner')

# Sort sequentially so time moves forward for each patient
final_dataset.sort_values(by=['stay_id', 'time_hour'], inplace=True)

# ==========================================
# 7. EXPORT TO CSV
# ==========================================
output_filename = "chronos_sentinel_timeseries_master.csv"
final_dataset.to_csv(output_filename, index=False)

print(f"Success! Master dataset generated: {output_filename}")
print(f"Shape: {final_dataset.shape} (Rows: Hours of patient data, Columns: Physiological Features)")

Loading datasets...
Extracting and standardizing tables...
Combining all temporal events...
Synchronizing timelines (Hourly Resampling)...
Pivoting data to feature matrix...


C:\Users\anjee\AppData\Local\Temp\ipykernel_30412\1706788226.py:50: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  all_events['time_hour'] = all_events['timestamp'].dt.floor('H')


Adding ICU Stay boundaries...
Success! Master dataset generated: chronos_sentinel_timeseries_master.csv
Shape: (12387, 747) (Rows: Hours of patient data, Columns: Physiological Features)


In [26]:
print("Loading datasets...")
chartevents = pd.read_csv('chartevents.csv', low_memory=False)
inputevents = pd.read_csv('inputevents.csv', low_memory=False)
outputevents = pd.read_csv('outputevents.csv', low_memory=False)
icustays = pd.read_csv('icustays.csv', low_memory=False)
d_items = pd.read_csv('d_items.csv', low_memory=False)

print("Extracting and standardizing tables...")


df_chart = chartevents[['subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'valuenum']].copy()
df_chart.rename(columns={'charttime': 'timestamp', 'valuenum': 'event_value'}, inplace=True)
df_chart['event_type'] = 'Vital'

# The specific Item IDs for core hemodynamics (Heart Rate, MAP, SpO2, Respiratory Rate, Temp, Glucose, Lactic Acid)
# This entirely removes IV gauges, dressings, and administrative notes.
core_vitals_ids = [220045, 220052, 220210, 220277, 225664, 225668, 223761]

# Apply the filter to chartevents
df_chart = chartevents[chartevents['itemid'].isin(core_vitals_ids)]

# B. Input Events (IV Fluids, Vasopressors, Meds)
# Use 'starttime' as the event time, and 'amount' or 'rate' as the value
df_input = inputevents[['subject_id', 'hadm_id', 'stay_id', 'starttime', 'itemid', 'amount']].copy()
df_input.rename(columns={'starttime': 'timestamp', 'amount': 'event_value'}, inplace=True)
df_input['event_type'] = 'Input'

# C. Output Events (Urine Output, Drains)
df_output = outputevents[['subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'value']].copy()
df_output.rename(columns={'charttime': 'timestamp', 'value': 'event_value'}, inplace=True)
df_output['event_value'] = pd.to_numeric(df_output['event_value'], errors='coerce') # Force text to NaN
df_output['event_type'] = 'Output'

# ==========================================
# 3. COMBINE ALL EVENTS & MAP LABELS
# ==========================================
print("Combining all temporal events...")

# Stack them vertically into one massive long-format dataframe
all_events = pd.concat([df_chart, df_input, df_output], ignore_index=True)

# Drop any rows with missing values or missing timestamps to clean the data
all_events.dropna(subset=['event_value', 'timestamp'], inplace=True)

# Merge with d_items to get human-readable names (Explainable UI requirement)
all_events = all_events.merge(d_items[['itemid', 'label']], on='itemid', how='left')

# ==========================================
# 4. TIME-SERIES SYNCHRONIZATION (Hourly Bins)
# ==========================================
print("Synchronizing timelines (Hourly Resampling)...")

# Convert timestamp to datetime object
all_events['timestamp'] = pd.to_datetime(all_events['timestamp'])

# ROUND DOWN to the nearest hour (e.g., 10:45 AM becomes 10:00 AM)
# This allows us to sync a blood pressure taken at 10:15 with an IV drip started at 10:45
all_events['time_hour'] = all_events['timestamp'].dt.floor('H')

# ==========================================
# 5. PIVOT TO THE FINAL ML MATRIX
# ==========================================
print("Pivoting data to feature matrix...")

# Group by patient stay and the specific hour, then pivot the labels into columns
# We use 'mean' for vitals and 'sum' for inputs/outputs is generally better, 
# but 'mean' works as a universal aggregator for the baseline.
time_series_matrix = all_events.pivot_table(
    index=['subject_id', 'hadm_id', 'stay_id', 'time_hour'],
    columns='label',
    values='event_value',
    aggfunc='mean'
).reset_index()

# ==========================================
# 6. MERGE WITH ICU STAY CONTEXT
# ==========================================
print("Adding ICU Stay boundaries...")

# Only keep the necessary context from icustays
icu_context = icustays[['stay_id', 'intime', 'outtime', 'los']]

final_dataset = time_series_matrix.merge(icu_context, on='stay_id', how='inner')

# Sort sequentially so time moves forward for each patient
final_dataset.sort_values(by=['stay_id', 'time_hour'], inplace=True)

# ==========================================
# 7. EXPORT TO CSV
# ==========================================
output_filename = "chronos_sentinel_timeseries_master.csv"
final_dataset.to_csv(output_filename, index=False)

print(f"Success! Master dataset generated: {output_filename}")
print(f"Shape: {final_dataset.shape} (Rows: Hours of patient data, Columns: Physiological Features)")

Loading datasets...
Extracting and standardizing tables...
Combining all temporal events...
Synchronizing timelines (Hourly Resampling)...
Pivoting data to feature matrix...
Adding ICU Stay boundaries...


C:\Users\anjee\AppData\Local\Temp\ipykernel_30412\3236135528.py:58: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  all_events['time_hour'] = all_events['timestamp'].dt.floor('H')


Success! Master dataset generated: chronos_sentinel_timeseries_master.csv
Shape: (9544, 204) (Rows: Hours of patient data, Columns: Physiological Features)


In [37]:
df = pd.read_csv('chronos_sentinel_timeseries_master.csv')
df.shape

(9544, 204)

In [32]:
df.iloc[:,4 : 25].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9544 entries, 0 to 9543
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   ACD-A Citrate (1000ml)         32 non-null     float64
 1   Acetaminophen-IV               166 non-null    float64
 2   Acyclovir                      9 non-null      float64
 3   Albumin 25%                    51 non-null     float64
 4   Albumin 5%                     50 non-null     float64
 5   Alteplase (TPA)                1 non-null      float64
 6   Amino Acids                    3 non-null      float64
 7   Aminocaproic acid (Amicar)     8 non-null      float64
 8   Amiodarone                     27 non-null     float64
 9   Amiodarone 450/250             3 non-null      float64
 10  Amiodarone 600/500             56 non-null     float64
 11  Ampicillin                     34 non-null     float64
 12  Ampicillin/Sulbactam (Unasyn)  13 non-null     f

In [33]:
# Define lists of string keywords to search for in the medication labels
antiarrhythmics = ['Amiodarone', 'Atropine', 'Lidocaine']
volume_expanders = ['Albumin']
vasopressors = ['Norepinephrine', 'Epinephrine', 'Vasopressin', 'Dopamine']

# Function to categorize the medications
def categorize_medication(label):
    if type(label) != str: return 'Ignore'
    if any(drug in label for drug in antiarrhythmics): return 'Antiarrhythmic_Meds'
    if any(drug in label for drug in volume_expanders): return 'Volume_Expanders'
    if any(drug in label for drug in vasopressors): return 'Vasopressor_Meds'
    return 'Ignore' # Drops Tylenol, Boost, Antibiotics, etc.

# Apply this to your input events data (Assuming you merged with d_items to get 'label')
all_events['broad_category'] = all_events['label'].apply(categorize_medication)

# Filter out the 'Ignore' rows
critical_inputs = all_events[all_events['broad_category'] != 'Ignore'].copy()

# Now, instead of using the highly specific 'label' for your pivot columns, 
# use 'broad_category'. 
# This rolls all "Amiodarone 450" and "Atropine" into one dense column called "Antiarrhythmic_Meds"

In [34]:
critical_inputs.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1270 entries, 132 to 20403
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   subject_id      1270 non-null   int64         
 1   hadm_id         1270 non-null   int64         
 2   stay_id         1270 non-null   int64         
 3   caregiver_id    0 non-null      float64       
 4   charttime       0 non-null      object        
 5   storetime       0 non-null      object        
 6   itemid          1270 non-null   int64         
 7   value           0 non-null      object        
 8   valuenum        0 non-null      float64       
 9   valueuom        0 non-null      object        
 10  warning         0 non-null      float64       
 11  timestamp       1270 non-null   datetime64[ns]
 12  event_value     1270 non-null   float64       
 13  event_type      1270 non-null   object        
 14  label           1270 non-null   object        
 15  time_h

In [36]:
critical_inputs.head()

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning,timestamp,event_value,event_type,label,time_hour,broad_category
132,10005817,20626031,32604416,NaN,NaN,NaN,220864,NaN,NaN,NaN,NaN,2132-12-16 02:41:00,499.999981,Input,Albumin 5%,2132-12-16 02:00:00,Volume_Expanders
133,10005817,20626031,32604416,NaN,NaN,NaN,220864,NaN,NaN,NaN,NaN,2132-12-17 12:00:00,249.999990,Input,Albumin 5%,2132-12-17 12:00:00,Volume_Expanders
134,10005817,20626031,32604416,NaN,NaN,NaN,220864,NaN,NaN,NaN,NaN,2132-12-15 18:00:00,499.999981,Input,Albumin 5%,2132-12-15 18:00:00,Volume_Expanders
135,10005817,20626031,32604416,NaN,NaN,NaN,220864,NaN,NaN,NaN,NaN,2132-12-15 15:06:00,499.999981,Input,Albumin 5%,2132-12-15 15:00:00,Volume_Expanders
325,10020740,25826145,32145159,NaN,NaN,NaN,220862,NaN,NaN,NaN,NaN,2150-06-04 02:15:00,49.999999,Input,Albumin 25%,2150-06-04 02:00:00,Volume_Expanders


In [39]:
print("Loading datasets...")
chartevents = pd.read_csv('chartevents.csv', low_memory=False)
inputevents = pd.read_csv('inputevents.csv', low_memory=False)
outputevents = pd.read_csv('outputevents.csv', low_memory=False)
icustays = pd.read_csv('icustays.csv', low_memory=False)
d_items = pd.read_csv('d_items.csv', low_memory=False)

print("Extracting and standardizing tables...")

df_chart = chartevents[['subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'valuenum']].copy()
df_chart.rename(columns={'charttime': 'timestamp', 'valuenum': 'event_value'}, inplace=True)
df_chart['event_type'] = 'Vital'

# B. Input Events (IV Fluids, Vasopressors, Meds)
# Use 'starttime' as the event time, and 'amount' or 'rate' as the value
df_input = inputevents[['subject_id', 'hadm_id', 'stay_id', 'starttime', 'itemid', 'amount']].copy()
df_input.rename(columns={'starttime': 'timestamp', 'amount': 'event_value'}, inplace=True)
df_input['event_type'] = 'Input'

# C. Output Events (Urine Output, Drains)
df_output = outputevents[['subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'value']].copy()
df_output.rename(columns={'charttime': 'timestamp', 'value': 'event_value'}, inplace=True)
df_output['event_value'] = pd.to_numeric(df_output['event_value'], errors='coerce') # Force text to NaN
df_output['event_type'] = 'Output'

# 1. Ensure you extract ONLY the standardized columns before combining
# This prevents the "ghost" columns (0 non-nulls) from appearing
df_chart_clean = df_chart[['subject_id', 'hadm_id', 'stay_id', 'timestamp', 'itemid', 'event_value', 'event_type']].copy()
df_input_clean = df_input[['subject_id', 'hadm_id', 'stay_id', 'timestamp', 'itemid', 'event_value', 'event_type']].copy()
df_output_clean = df_output[['subject_id', 'hadm_id', 'stay_id', 'timestamp', 'itemid', 'event_value', 'event_type']].copy()

# 2. Combine them safely
all_events = pd.concat([df_chart_clean, df_input_clean, df_output_clean], ignore_index=True)
all_events.dropna(subset=['event_value', 'timestamp'], inplace=True)
all_events = all_events.merge(d_items[['itemid', 'label']], on='itemid', how='left')

# 3. The SMART Categorization Function
def categorize_all_events(row):
    label = str(row['label'])
    event_type = row['event_type']
    
    # If it's a vital sign or urine output, KEEP its exact label!
    if event_type == 'Vital' or event_type == 'Output':
        return label 
        
    # If it's an IV Medication (Input), group the important ones and drop the rest
    elif event_type == 'Input':
        antiarrhythmics = ['Amiodarone', 'Atropine', 'Lidocaine']
        volume_expanders = ['Albumin']
        vasopressors = ['Norepinephrine', 'Epinephrine', 'Vasopressin', 'Dopamine']
        
        if any(drug in label for drug in antiarrhythmics): return 'Antiarrhythmic_Meds'
        if any(drug in label for drug in volume_expanders): return 'Volume_Expanders'
        if any(drug in label for drug in vasopressors): return 'Vasopressor_Meds'
        
        # If it's a drug but not in our critical lists, label it for deletion
        return 'Drop_This_Med'

# Apply the smart categorization
all_events['final_feature_name'] = all_events.apply(categorize_all_events, axis=1)

# 4. Filter out ONLY the useless medications
critical_events = all_events[all_events['final_feature_name'] != 'Drop_This_Med'].copy()

# Print the shape to verify! It should be hundreds of thousands of rows now.
print(f"Corrected dataset size: {critical_events.shape}")

Loading datasets...
Extracting and standardizing tables...
Corrected dataset size: (268106, 9)


In [42]:
critical_events.info()

<class 'pandas.core.frame.DataFrame'>
Index: 268106 entries, 0 to 287239
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   subject_id          268106 non-null  int64  
 1   hadm_id             268106 non-null  int64  
 2   stay_id             268106 non-null  int64  
 3   timestamp           268106 non-null  object 
 4   itemid              268106 non-null  int64  
 5   event_value         268106 non-null  float64
 6   event_type          268106 non-null  object 
 7   label               268106 non-null  object 
 8   final_feature_name  268106 non-null  object 
dtypes: float64(1), int64(4), object(4)
memory usage: 20.5+ MB
